In [2]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, RegressorMixin, clone
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, HistGradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor


# Charger les données
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# Séparer la colonne cible de train
y_train = train['SalePrice']
train = train.drop(columns=['SalePrice'])

# Identifiant de test
test_ids = test['Id']

# Identifier automatiquement les colonnes numériques et catégoriques
numeric_features = train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = train.select_dtypes(include=['object']).columns

# Pipeline pour les colonnes numériques : imputation + scaling
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# Pipeline pour les colonnes catégoriques : imputation + one-hot encoding
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Combiner les transformations numériques et catégoriques
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Définir une classe pour la moyenne des modèles
class AveragingModels(BaseEstimator, RegressorMixin):
    def __init__(self, models):
        self.models = models
    
    def fit(self, X, y):
        self.models_ = [clone(x) for x in self.models]  # Clone les modèles
        for model in self.models_:
            model.fit(X, y)
        return self

    def predict(self, X):
        predictions = np.column_stack([model.predict(X) for model in self.models_])
        return np.mean(predictions, axis=1)  # Moyenne des prédictions

# Initialiser les modèles que tu veux combiner
ENet = ElasticNet(alpha=0.00005, l1_ratio=0.9, max_iter=3000, random_state=3)
lasso = Lasso(alpha=0.0001, max_iter=3000, random_state=1)
KRR = KernelRidge(alpha=0.6, kernel='polynomial', degree=2, coef0=2.5)
GBoost = GradientBoostingRegressor(n_estimators=3000, learning_rate=0.05,
                                  max_depth=4, max_features='sqrt',
                                  min_samples_leaf=15, min_samples_split=10, 
                                  loss='huber', random_state=5)
RF = RandomForestRegressor(n_estimators=1000, max_depth=10, random_state=42)
XGB = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=4, random_state=42)
LGBM = LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=4, random_state=42)
CAT = CatBoostRegressor(iterations=1000, learning_rate=0.05, depth=4, random_state=42, silent=True)

# Créer un modèle d'ensemble avec Averaging
averaging_model = AveragingModels(models=[ENet, lasso, KRR, GBoost, RF,XGB,KRR,CAT])

# Pipeline complet avec le préprocesseur et le modèle d'ensemble
model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', averaging_model)
])

# Entraîner le modèle avec validation croisée
n_folds = 5
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
rmse = np.sqrt(-cross_val_score(model_pipeline, train, y_train, scoring='neg_mean_squared_error', cv=kf))
print(f'Averaged RMSE score: {rmse.mean():.5f}')

# Entraîner le modèle d'ensemble sur l'ensemble du dataset
model_pipeline.fit(train, y_train)

# Faire des prédictions sur le dataset de test
predictions = model_pipeline.predict(test)

# Ajouter les prédictions au DataFrame pour les sauvegarder
predictions_df = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': predictions
})

# Sauvegarder en .csv
predictions_df.to_csv('predictionsHMA.csv', index=False)


c:\Users\arnov\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:658: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 224104673277.16504, tolerance: 696659484.3571944
  model = cd_fast.sparse_enet_coordinate_descent(
c:\Users\arnov\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:658: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 110681926774.29163, tolerance: 696659484.3571944
  model = cd_fast.sparse_enet_coordinate_descent(
c:\Users\arnov\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:658: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 245572029380.16928, tolerance: 721310725.7925861
  model = cd_fast.sparse_enet_coordinate_desc

KeyboardInterrupt: 